In [1]:
import numpy as np
import pandas as pd
import re
import pickle
import json
import time
import os
import tensorflow as tf
from tensorflow.keras.layers import Embedding, LSTM, Dense, Layer, Input, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
tf.random.set_seed(42)
np.random.seed(42)

In [3]:
df = pd.read_csv("Dataset_English_Hindi.csv")
df.drop_duplicates(inplace=True)
df = df.dropna()
df["English"] = df["English"].astype(str)
df["Hindi"]   = df["Hindi"].astype(str)
print(f"Total pairs: {len(df)}")

Total pairs: 127375


In [4]:
def clean_english(sentence):
    sentence = sentence.lower().strip()
    sentence = re.sub(r'[^a-zA-Z0-9\s]', '', sentence)
    sentence = re.sub(r'https?://\S+|www\.\S+', '', sentence)
    emoji_pattern = re.compile("["
                           u"\U0001F600-\U0001F64F"  # emoticons
                           u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                           u"\U0001F680-\U0001F6FF"  # transport & map symbols
                           u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                           u"\U00002702-\U000027B0"
                           u"\U000024C2-\U0001F251"
                           "]+", flags=re.UNICODE)
    sentence = emoji_pattern.sub(r'', sentence)
    return sentence.strip()

In [5]:
def clean_hindi(sentence):
    sentence = sentence.strip()
    sentence = re.sub(r'[^\u0900-\u097F\s]', '', sentence)
    sentence = re.sub(r'https?://\S+|www\.\S+', '', sentence)
    return sentence.strip()

In [6]:
df["English"] = df["English"].apply(clean_english)
df["Hindi"]   = df["Hindi"].apply(clean_hindi)

In [7]:
# Filter by sentence length for efficient training


MAX_ENG_WORDS = 25
MAX_HIN_WORDS = 30


mask = (
    df["English"].str.split().str.len().between(5, MAX_ENG_WORDS) &
    df["Hindi"].str.split().str.len().between(5, MAX_HIN_WORDS)
)
df = df[mask].reset_index(drop=True)
print(f"Pairs after length filter: {len(df)}")

Pairs after length filter: 88576


In [8]:
# Add <start> / <end> tokens to Hindi
df["Hindi_in"]  = "<sos> " + df["Hindi"]          # decoder input
df["Hindi_out"] = df["Hindi"] + " <eos>"          # decoder target

In [9]:
# English tokenizer
eng_tok = Tokenizer(filters="", lower=True, oov_token="<OOV>")
eng_tok.fit_on_texts(df["English"])
ENG_VOCAB = len(eng_tok.word_index) + 1

In [10]:
# Hindi tokenizer  (case-sensitive for Devanagari)
hin_tok = Tokenizer(filters="", lower=False, oov_token="<OOV>")
hin_tok.fit_on_texts(df["Hindi_in"].tolist() + df["Hindi_out"].tolist())
HIN_VOCAB = len(hin_tok.word_index) + 1

In [11]:
print(f"English vocab: {ENG_VOCAB}  |  Hindi vocab: {HIN_VOCAB}")

English vocab: 52611  |  Hindi vocab: 54090


In [12]:
df_enc_inp = pad_sequences(
    eng_tok.texts_to_sequences(df["English"]),
    maxlen = MAX_ENG_WORDS, padding="post"
)
df_dec_inp = pad_sequences(
    hin_tok.texts_to_sequences(df["Hindi_in"]),
    maxlen = MAX_HIN_WORDS + 1, padding="post"
)
df_dec_out = pad_sequences(
    hin_tok.texts_to_sequences(df["Hindi_out"]),
    maxlen = MAX_HIN_WORDS + 1, padding="post"
)

In [13]:
ENG_SEQ_LEN = df_enc_inp.shape[1]
HIN_SEQ_LEN = df_dec_inp.shape[1]
print(f"Encoder seq len: {ENG_SEQ_LEN}  |  Decoder seq len: {HIN_SEQ_LEN}")
print(f"df_enc_inp shape: {df_enc_inp.shape}")
print(f"df_dec_out shape: {df_dec_out.shape}")

Encoder seq len: 25  |  Decoder seq len: 31
df_enc_inp shape: (88576, 25)
df_dec_out shape: (88576, 31)


In [14]:
#Model Hyper Parameters
EMBED_DIM = 512
BATCH_SIZE = 32
EPOCHS = 50

In [15]:
#as Keras doesn't have an in-built Bahdanau Attention Class
#here is a custom class for the same
class BahdanauAttention(Layer):
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.W1 = Dense(units, name="attn_W1")   # projects encoder states
        self.W2 = Dense(units, name="attn_W2")   # projects decoder states
        self.V  = Dense(1,     name="attn_V")    # scalar score per enc step

    def call(self, query, values):
        # query  : (batch, dec_len, EMBED_DIM)  — decoder hidden states
        # values : (batch, enc_len, EMBED_DIM)  — encoder hidden states

        # Expand for broadcasting over both sequence axes
        query_exp  = tf.expand_dims(query,  2)   # (batch, dec_len, 1,       EMBED_DIM)
        values_exp = tf.expand_dims(values, 1)   # (batch, 1,       enc_len, EMBED_DIM)

        # score : (batch, dec_len, enc_len)
        score = self.V(tf.nn.tanh(self.W1(values_exp) + self.W2(query_exp)))
        score = tf.squeeze(score, axis=-1)

        # Softmax over the encoder-length axis
        attn_weights = tf.nn.softmax(score, axis=-1)   # (batch, dec_len, enc_len)

        # Weighted sum of encoder states → context
        context = tf.matmul(attn_weights, values)      # (batch, dec_len, EMBED_DIM)
        return context, attn_weights

    def get_config(self):
        config = super().get_config()
        config.update({"units": self.units})
        return config

In [16]:
#Encoder Model
enc_input = Input(shape=(ENG_SEQ_LEN,), name="enc_input")

enc_embedding_layer = Embedding(ENG_VOCAB, EMBED_DIM, name="enc_embedding") #shared layer
enc_embed = enc_embedding_layer(enc_input)

enc_lstm = LSTM(EMBED_DIM, return_sequences=True, return_state=True, name="enc_lstm") #shared layer
enc_out, enc_h, enc_c = enc_lstm(enc_embed)

enc_model = Model(enc_input, [enc_out, enc_h, enc_c])

In [17]:
#Decoder Model
dec_input = Input(shape=(HIN_SEQ_LEN,), name="dec_input")

dec_embedding_layer = Embedding(HIN_VOCAB, EMBED_DIM, name="dec_embedding") #shared layer
dec_embed = dec_embedding_layer(dec_input)

dec_lstm = LSTM(EMBED_DIM, return_sequences=True, return_state=True, name="dec_lstm") #shared layer
dec_out, dec_h, dec_c = dec_lstm(dec_embed, initial_state=[enc_h, enc_c])

#dec_dense  = Dense(HIN_VOCAB, activation='softmax', name="dec_dense")(dec_out)

In [18]:
#Bahdanau Attention Model
attn_layer = BahdanauAttention(EMBED_DIM, name="bahdanau_attention") #shared layer
context_vector, _ = attn_layer(dec_out, enc_out)

concat_layer = Concatenate(axis=-1, name="context_concat") #shared layer
dec_combined = concat_layer([dec_out, context_vector])

In [19]:
#Dense Layer
dec_dense_layer = Dense(HIN_VOCAB, activation='softmax', name="dec_dense") #shared layer
dec_dense = dec_dense_layer(dec_combined)

In [20]:
#Model
model = Model([enc_input, dec_input], dec_dense)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 enc_input (InputLayer)         [(None, 25)]         0           []                               
                                                                                                  
 dec_input (InputLayer)         [(None, 31)]         0           []                               
                                                                                                  
 enc_embedding (Embedding)      (None, 25, 512)      26936832    ['enc_input[0][0]']              
                                                                                                  
 dec_embedding (Embedding)      (None, 31, 512)      27694080    ['dec_input[0][0]']              
                                                                                            

In [21]:
from tensorflow.keras.callbacks import EarlyStopping
early_stopping = EarlyStopping(monitor='accuracy', patience=5, restore_best_weights=True)

In [22]:
#Training
history = model.fit(
    x=[df_enc_inp, df_dec_inp],
    y=df_dec_out,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    callbacks = [early_stopping]
)

Epoch 1/50
2768/2768 [==============================] - 286s 102ms/step - loss: 2.7937 - accuracy: 0.6148
Epoch 2/50
2768/2768 [==============================] - 285s 103ms/step - loss: 1.8640 - accuracy: 0.6913
Epoch 3/50
2768/2768 [==============================] - 286s 103ms/step - loss: 1.2655 - accuracy: 0.7534
Epoch 4/50
2768/2768 [==============================] - 287s 104ms/step - loss: 0.8760 - accuracy: 0.8106
Epoch 5/50
2768/2768 [==============================] - 287s 104ms/step - loss: 0.6533 - accuracy: 0.8504
Epoch 6/50
2768/2768 [==============================] - 287s 104ms/step - loss: 0.5044 - accuracy: 0.8798
Epoch 7/50
2768/2768 [==============================] - 287s 104ms/step - loss: 0.3936 - accuracy: 0.9035
Epoch 8/50
2768/2768 [==============================] - 287s 104ms/step - loss: 0.3107 - accuracy: 0.9219
Epoch 9/50
2768/2768 [==============================] - 287s 104ms/step - loss: 0.2505 - accuracy: 0.9359
Epoch 10/50
2768/2768 [=======================

In [23]:
#model.save("my_model.keras")
#model.save_weights("my_model.weights.h5")
#with open("my_model_architecture.json", 'w') as f:
#    f.write(model.to_json())

history_df = pd.DataFrame(history.history)
history_df.index = range(1, len(history_df) + 1)   # epochs start from 1
history_df.index.name = "epoch"
history_df.to_csv("training_history.csv")

In [25]:
# Inference — Encoder Model
encoder_inf_model = Model(enc_input, [enc_out, enc_h, enc_c])

# Inference — Decoder Model (one token at a time)
dec_inf_input = Input(shape=(1,), name="dec_inf_input")
dec_state_inp_h = Input(shape=(EMBED_DIM,), name="dec_state_inp_h")
dec_state_inp_c = Input(shape=(EMBED_DIM,), name="dec_state_inp_c")
enc_outputs_inp = Input(shape=(ENG_SEQ_LEN, EMBED_DIM), name="enc_outputs_inp")

dec_inf_embed = dec_embedding_layer(dec_inf_input)    # (batch, 1, EMBED_DIM)

dec_inf_out, dec_inf_h, dec_inf_c = dec_lstm(
    dec_inf_embed, initial_state=[dec_state_inp_h, dec_state_inp_c]
)  # dec_inf_out : (batch, 1, EMBED_DIM)

# Attend over the full encoder output sequence at every decode step
context_inf, attn_inf = attn_layer(dec_inf_out, enc_outputs_inp)
# context_inf : (batch, 1, EMBED_DIM)

dec_inf_combined = concat_layer([dec_inf_out, context_inf])
dec_inf_dense    = dec_dense_layer(dec_inf_combined)
# dec_inf_dense : (batch, 1, HIN_VOCAB)

decoder_inf_model = Model(
    inputs  = [dec_inf_input, dec_state_inp_h, dec_state_inp_c, enc_outputs_inp],
    outputs = [dec_inf_dense, dec_inf_h, dec_inf_c]
)

In [26]:
#Translate
hin_idx2word = {idx: word for word, idx in hin_tok.word_index.items()}
sos_token    = hin_tok.word_index["<sos>"]
eos_token    = hin_tok.word_index["<eos>"]

In [27]:
def translate(english_sentence, max_len=MAX_HIN_WORDS):
    # Step 1: Clean and pad English input
    eng_clean = clean_english(english_sentence)
    eng_seq   = eng_tok.texts_to_sequences([eng_clean])
    eng_pad   = pad_sequences(eng_seq, maxlen=ENG_SEQ_LEN, padding="post")

    # Step 2: Encode → all encoder hidden states + initial decoder states
    enc_outs, h, c = encoder_inf_model.predict(eng_pad, verbose=0)

    # Step 3: Start decoding with <sos>
    target_token = np.zeros((1, 1))
    target_token[0, 0] = sos_token

    translated = []

    # Step 4: Decode one token at a time, attending over enc_outs each step
    for _ in range(max_len):
        pred, h, c = decoder_inf_model.predict(
            [target_token, h, c, enc_outs], verbose=0
        )

        predicted_id   = np.argmax(pred[0, 0, :])
        predicted_word = hin_idx2word.get(predicted_id, "")

        if predicted_id == eos_token or predicted_word == "":
            break

        translated.append(predicted_word)
        target_token[0, 0] = predicted_id   # feed output as next input

    return " ".join(translated)

In [28]:
#Test
test_sentences = [
    "How are you doing today",
    "I want to eat food",
    "The weather is very cold outside",
    "She is going to the market",
    "My name is Rahul",
    "I love my country",
    "He is a very good student",
    "Please open the door",
    "I am feeling very tired today",
    "The dog is running in the park",
    "Can you help me please",
    "I want to drink water",
    "She loves to read books",
    "The sun rises in the east",
    "I go to school every day",
    "He does not like cold weather",
    "We are going to the temple",
    "The train is very late today",
    "I want to sleep now",
    "My mother cooks very delicious food"
]

for s in test_sentences:
    print(f"English : {s}")
    print(f"Hindi   : {translate(s)}")
    print("-" * 40)

English : How are you doing today
Hindi   : आज हम लोग आज कर रहे हैं
----------------------------------------
English : I want to eat food
Hindi   : मैं खाना खाने के लिए खाना खाने दो कदम देना चाहता हूँ
----------------------------------------
English : The weather is very cold outside
Hindi   : मौसम बाहर की तरफ आती है।
----------------------------------------
English : She is going to the market
Hindi   : और बाज़ार जा रही है कि वह बातचीत कर रहा है।
----------------------------------------
English : My name is Rahul
Hindi   : मेरा नाम है इन सबसे पसंदीता मौसम है।
----------------------------------------
English : I love my country
Hindi   : मुझे मेरी देश बढ़िया पसंद है
----------------------------------------
English : He is a very good student
Hindi   : वह बहुत अच्छे विद्यार्थी है।
----------------------------------------
English : Please open the door
Hindi   : कृपया दरवाज़ा खोल पड़ते हैं
----------------------------------------
English : I am feeling very tired today
Hindi   : आज मेरे 